# Context length thật của model local: needle in a haystack

Model card của Qwen2.5-3B ghi context 32 768 token. Bạn đo xem `qwen2.5:3b` chạy qua Ollama trên máy bạn **thật sự**
tìm được một dữ kiện nhỏ (needle) giấu trong văn bản dài (haystack) tới đâu: theo độ dài prompt, theo vị trí của
dữ kiện, và theo cấu hình `num_ctx`. Kết quả là một bản đồ "dùng được tới đâu" và `REPORT.md` cho đồng nghiệp.

**Cách làm bài**
- Hai hàm trong cell chấm điểm có test. Phần thí nghiệm (cell `explore`) là của bạn: khung dưới đây chỉ gợi ý.
- Cần Ollama với `qwen2.5:3b`, bộ `wikipedia-vi` (filler) và tokenizer `Qwen/Qwen2.5-0.5B-Instruct` (cùng tokenizer
  với Qwen2.5-3B, vài MB) để đếm token.
- Prompt 12 000 token trên GPU 4 GB mất cỡ chục giây tới vài phút mỗi lần: cache từng kết quả ra tệp.
- Chấm: `uv run pytest tracks/ai-engineer/modules/09-llms-and-open-models/exercises/advanced/02-needle-in-haystack -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import re
from collections.abc import Callable, Sequence

## 1. Hai hàm được chấm

`build_haystack(fillers, needle, depth, target_tokens, count_tokens) -> str`:

- Lấy lần lượt các đoạn `fillers` theo thứ tự (hết thì quay lại đoạn đầu), cộng dồn `count_tokens(đoạn)` của
  **từng đoạn**, dừng ngay khi tổng `>= target_tokens`. Không đếm needle, không đếm lại cả văn bản.
- Chèn `needle` thành một đoạn riêng ở vị trí `round(depth * số_đoạn_filler)`: `depth = 0` là đầu, `1` là cuối.
- Nối các đoạn bằng `"\n\n"`. `depth` ngoài `[0, 1]` hoặc `fillers` rỗng: `ValueError`.

`needle_found(reply, answer) -> bool`: `answer` là một dãy chữ số. Bỏ dấu phân cách nghìn trong `reply` (dấu cách,
dấu chấm hoặc phẩy nằm giữa hai chữ số, trước đúng 3 chữ số cuối nhóm: `7.429`, `7 429`), rồi kiểm `answer` có
xuất hiện như một số **nguyên vẹn** (không dính chữ số khác hai bên: `17429` không tính).

In [ ]:
def build_haystack(fillers: Sequence[str], needle: str, depth: float, target_tokens: int,
                   count_tokens: Callable[[str], int]) -> str:
    """Văn bản dài >= target_tokens token filler, needle là một đoạn ở vị trí round(depth * số đoạn)."""
    # TODO
    raise NotImplementedError


def needle_found(reply: str, answer: str) -> bool:
    """answer (dãy số) có trong reply như một số nguyên vẹn, chấp nhận dấu phân cách nghìn."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: filler từ Wikipedia tiếng Việt, tokenizer để đếm token
import json
import sys
import time
from pathlib import Path

import httpx
import pyarrow.parquet as pq
from transformers import AutoTokenizer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

WIKI = ensure("wikipedia-vi")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
count = lambda t: len(tok(t, add_special_tokens=False)["input_ids"])
batch = next(pq.ParquetFile(WIKI / "20231101.vi" / "train-00002-of-00004.parquet")
             .iter_batches(batch_size=2000, columns=["text"]))
fillers = [p.strip() for t in batch.column("text").to_pylist() for p in t.split("\n") if 200 <= len(p.strip()) <= 1500]
print(len(fillers), "đoạn filler")

In [ ]:
# explore: TODO lưới thí nghiệm
# Gợi ý: độ dài {1 000, 3 000, 6 000, 12 000} token × depth {0, 0.25, 0.5, 0.75, 1} × num_ctx {mặc định, 16 384}.
# Mỗi ô một mã số khác nhau, temperature 0, ghi prompt_eval_count Ollama trả về, câu trả lời, đúng/sai, thời gian.
# Cuối cùng gọi lại model một lần với num_ctx mặc định để trả server dùng chung về cấu hình nhẹ.

In [ ]:
# explore: TODO bảng/heatmap đúng-sai theo (độ dài, depth) cho từng num_ctx; so prompt_eval_count với số token bạn đếm

## 2. Báo cáo

Viết `REPORT.md` (cùng thư mục) theo khung có sẵn, rồi làm phần trắc nghiệm của bài trong app.